# Model 4 — Hydro-TEM: temporal embedded ensemble

**Run all without editing code.** This notebook embeds the complete implementation,
finds the attached tabular dataset, and downloads the public dataset automatically
if it is absent. Use a Kaggle **GPU** accelerator; enable **Internet** if the dataset
or a dependency is missing. SAR images are not required. No GitHub clone is used.

The previous results favour numerical embeddings and BCE: Model 2 AP 0.8355;
Model 3's graph AP 0.7527; LightGBM AP 0.8496. These are historical context,
not scores for this new experiment. **Model 4 has not yet been measured.**

Model 4 combines train-quantile piecewise-linear numerical embeddings, a strong
current-day branch, gated lag/mean/change features over 14 days, terrain and
seasonality, and four jointly trained neural ensemble members per seed. Separate
dry/wet hazards handle onset versus persistence; cumulative hazards enforce
P(24h) ≤ P(48h) ≤ P(72h). Three independent seeds are averaged.

The implementation is inspired by [numerical embeddings](https://arxiv.org/abs/2203.05556)
and [efficient tabular ensembling](https://arxiv.org/abs/2410.24210), and is a custom
research architecture rather than an official TabM reproduction.


## Predeclared experiment

| Purpose | Dates |
|---|---|
| Training and fitted preprocessing | 2003–2017 |
| Checkpoint / early-stopping selection (next-day AP) | 2018–2019 |
| Probability calibration and decision thresholds | 2020 |
| Final evaluation | 2021–2024 |

Origins whose three-day targets cross any boundary are removed. Future targets
are rebuilt within nodes with a full three-day observation requirement. Historical
context from the preceding period remains allowed. No random split, test-tuned
threshold, test-selected seed, focal loss, or class oversampling is used.

**Comparators:** Model 2 architecture retrained with the same new inputs, split,
classification loss and seed count; LightGBM; discharge-percentile rule; persistence;
and a one-seed current-day-only Model 4 ablation. The control is not a rerun of
the old N5 configuration: preprocessing, loss auxiliaries and selection differ.
Compare the freshly generated rows. The ablation is exploratory because it has
one seed. Per-seed scores and paired 28-day block-bootstrap intervals are saved.

**Interpretation:** these labels are GloFAS discharge-Q98 exceedances, not surveyed
flood inundation. The processed dataset already contains interpolated/backfilled
weather/soil and reanalysis data. This notebook cannot undo that inherited
availability leakage. Earlier models have already exposed the test years, so a
positive result here still requires a new locked future/event holdout before a
confirmatory or operational claim. A Q98 percentile is not itself a fitted
two-year return level. No best-in-research or state-of-the-art result is promised.


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

assert Path('/kaggle/working').exists(), 'Run this notebook on Kaggle.'
required = {'torch':'torch', 'numpy':'numpy', 'pandas':'pandas',
            'sklearn':'scikit-learn', 'scipy':'scipy', 'pyarrow':'pyarrow',
            'lightgbm':'lightgbm', 'matplotlib':'matplotlib', 'joblib':'joblib'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle Settings, then Run All.'
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)

hits = sorted(Path('/kaggle/input').rglob('flood_dataset.parquet'))
hits = [p for p in hits if (p.parent/'nodes.csv').exists()]
if not hits:
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'kagglehub'], check=True)
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('uom230429e/sri-lanka-flood-tabular-graph-2003-2025'))
    hits = [p for p in downloaded.rglob('flood_dataset.parquet') if (p.parent/'nodes.csv').exists()]
if not hits:
    raise FileNotFoundError('The tabular input must contain flood_dataset.parquet and nodes.csv.')
if len(hits)>1:
    import hashlib
    fingerprints = {hashlib.sha256(p.read_bytes()).hexdigest() for p in hits}
    if len(fingerprints)>1:
        raise RuntimeError('Multiple different panel versions are attached; retain one intended tabular dataset.')
DATA_ROOT = hits[0].parent
print('Dataset:', DATA_ROOT)


In [ ]:
# Exact source snapshot: self-contained, including the Model 2 control.
import hashlib, json, sys
from pathlib import Path
CODE_ROOT = Path('/kaggle/working/model4_code')
SOURCES = {'models/model4/__init__.py': '"""Hydro-TEM: temporal embedded ensemble for discharge-exceedance forecasting."""\n', 'models/model4/workflow.py': '"""Self-contained Model 4 experiment, embedded in the Kaggle notebook.\n\nResearch candidate, not a claim of improvement. See docs/MODEL4.md.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport time\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import (average_precision_score, roc_auc_score,\n                             precision_recall_curve)\nfrom scipy.optimize import minimize\n\nfrom floodlib.schema import DYNAMIC_FEATURES, LOG1P_FEATURES, ZONES, POSITIONS\nfrom model2.config import MMFConfig\nfrom model2.model import MMFNet\n\n\n@dataclass\nclass Config:\n    lookback: int = 14\n    width: int = 128\n    members: int = 4\n    bins: int = 24\n    embedding: int = 8\n    dropout: float = 0.15\n    epochs: int = 80\n    patience: int = 12\n    batch_size: int = 1024\n    lr: float = 0.0005\n    weight_decay: float = 0.0001\n    grad_clip: float = 2.0\n    seeds: tuple = (0, 1, 2)\n    budget_hours: float = 7.5\n    bootstrap_draws: int = 500\n    max_far: float = 0.231\n\n\ndef json_save(path, value):\n    def clean(x):\n        if isinstance(x, dict):\n            return {str(k): clean(v) for k, v in x.items()}\n        if isinstance(x, (list, tuple)):\n            return [clean(v) for v in x]\n        if isinstance(x, np.ndarray):\n            return clean(x.tolist())\n        if isinstance(x, (float, np.floating)):\n            return float(x) if np.isfinite(x) else None\n        if isinstance(x, np.integer):\n            return int(x)\n        return x\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(clean(value), indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef torch_save(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(\'.tmp\')\n    torch.save(value, tmp)\n    tmp.replace(path)\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for part in iter(lambda: f.read(2**20), b\'\'):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef prepare(df, nodes, cfg):\n    """Dense daily panel; rebuild full-horizon targets and purge boundaries.\n\n    All normalisers/knots use training origins only. Historical context may\n    cross a split boundary; the future labels may not. No backward filling.\n    """\n    df = df.copy()\n    df[\'date\'] = pd.to_datetime(df.date).dt.normalize()\n    df = df.loc[df.date.between(\'2003-01-01\', \'2024-12-31\')]\n    if df.empty or df.duplicated([\'node_id\', \'date\']).any():\n        raise ValueError(\'Panel is empty or contains duplicate node/day keys.\')\n    required = [\'discharge\', \'discharge_pctl\', \'valid_sample\'] + DYNAMIC_FEATURES\n    absent = sorted(set(required) - set(df))\n    if absent:\n        raise ValueError(f\'Incomplete tabular dataset: missing {absent}\')\n    ids = sorted(df.node_id.unique())\n    dates = pd.date_range(df.date.min(), df.date.max(), freq=\'D\')\n    grid = pd.MultiIndex.from_product([dates, ids], names=[\'date\', \'node_id\'])\n    g = df.set_index([\'date\', \'node_id\']).reindex(grid)\n    T, N = len(dates), len(ids)\n    raw = g[DYNAMIC_FEATURES].to_numpy(np.float32, copy=True).reshape(T, N, -1)\n    raw[~np.isfinite(raw)] = np.nan\n    q = raw[..., DYNAMIC_FEATURES.index(\'discharge\')]\n    train_dates = dates <= \'2017-12-31\'\n    threshold = np.nanquantile(q[train_dates], .98, axis=0).astype(\'float32\')\n    if \'thr_high\' in g:\n        # The archived threshold defines the task; verify it against train Q.\n        stored = g[\'thr_high\'].to_numpy(np.float32).reshape(T, N)\n        archived = np.nanmedian(stored[train_dates], axis=0)\n        if not np.allclose(archived, threshold, rtol=1e-3, atol=1e-4):\n            raise ValueError(\'Archived high thresholds differ from 2003-2017 Q98; inspect dataset version.\')\n        threshold = archived\n    if not np.isfinite(threshold).all() or np.any(threshold <= 0):\n        raise ValueError(\'Missing or nonpositive per-node flood threshold.\')\n    state = np.where(np.isfinite(q), (q >= threshold).astype(float), np.nan)\n    future = np.full((T, N, 3), np.nan, np.float32)\n    for h in range(1, 4):\n        future[:-h, :, h-1] = state[h:]\n    y = np.maximum.accumulate(future, axis=-1)\n    y = np.concatenate([y, (y[..., :1] * (1-state[..., None]))], axis=-1)\n    # Separate model selection (2018-19), calibration (2020), and test (2021-24).\n    period = np.select([dates <= \'2017-12-31\', dates <= \'2019-12-31\',\n                        dates <= \'2020-12-31\'], [0, 1, 2], default=3)\n    valid = (g.valid_sample.fillna(0).to_numpy().reshape(T, N) > 0)\n    valid &= np.isfinite(future).all(-1) & np.isfinite(state)\n    for h in range(1, 4):\n        same = np.zeros(T, bool)\n        same[:-h] = period[:-h] == period[h:]\n        valid &= same[:, None]\n    valid[:cfg.lookback-1] = False\n    masks = {name: valid & (period[:, None] == i)\n             for i, name in enumerate([\'train\', \'select\', \'calibrate\', \'test\'])}\n    if any(not m.any() for m in masks.values()):\n        raise ValueError(\'All four chronological periods require usable samples.\')\n    # A directly interpretable causal feature: distance from flood threshold.\n    ratio = np.log(np.maximum(q, 1e-6) / threshold)[..., None]\n    raw = np.concatenate([raw, ratio], axis=-1)\n    feature_names = DYNAMIC_FEATURES + [\'log_q_over_train_q98\']\n    for i, name in enumerate(feature_names):\n        if name in LOG1P_FEATURES:\n            raw[..., i] = np.sign(raw[..., i]) * np.log1p(np.abs(raw[..., i]))\n    tr = raw[masks[\'train\']]\n    median = np.nanmedian(tr, axis=0)\n    median = np.where(np.isfinite(median), median, 0)\n    missing = ~np.isfinite(raw)\n    filled = np.where(missing, median, raw)\n    mean = filled[masks[\'train\']].mean(0)\n    std = filled[masks[\'train\']].std(0)\n    std = np.where(std > 1e-6, std, 1)\n    numeric = np.clip((filled-mean)/std, -12, 12).astype(\'float32\')\n    phase = 2*np.pi*(dates.dayofyear.to_numpy()-1)/365.25\n    season = np.broadcast_to(np.stack([np.sin(phase), np.cos(phase)], -1)[:, None], (T,N,2))\n    x = np.concatenate([numeric, missing.astype(\'float32\'), season,\n                        np.nan_to_num(state)[...,None]], -1).astype(\'float32\')\n    features = feature_names + [f\'{f}__missing\' for f in feature_names] + [\'doy_sin\',\'doy_cos\',\'current_flood_state\']\n    nd = nodes.set_index(\'node_id\').reindex(ids)\n    if nd[[\'elevation_m\', \'zone\', \'position\', \'basin\']].isna().any().any():\n        raise ValueError(\'nodes.csv must provide terrain metadata for every panel node.\')\n    drainage = np.log1p(np.nanmean(q[train_dates], axis=0))\n    cont = np.column_stack([nd.elevation_m.to_numpy(float), drainage])\n    sm, ss = cont.mean(0), cont.std(0).clip(1e-6)\n    s = np.column_stack([(cont-sm)/ss, *[(nd.zone == z).to_numpy(float) for z in ZONES],\n                         *[(nd.position == p).to_numpy(float) for p in POSITIONS]]).astype(\'float32\')\n    knots = np.quantile(numeric[masks[\'train\']], np.linspace(0,1,cfg.bins+1), axis=0).T\n    mismatch = {}\n    for h, col in enumerate([\'target_flood_1d\',\'target_flood_2d\',\'target_flood_3d\',\'target_onset_1d\']):\n        if col in g:\n            old = g[col].to_numpy(float).reshape(T,N)\n            check = valid & np.isfinite(old)\n            mismatch[col] = int(np.sum(old[check] != y[...,h][check]))\n    audit = {\'shape\': [T,N,x.shape[-1]], \'features\': features, \'numeric_features\': feature_names,\n             \'nodes\': ids, \'thresholds_q98\': threshold,\n             \'split_counts\': {k:int(v.sum()) for k,v in masks.items()},\n             \'positive_rates\': {k:y[v].mean(0) for k,v in masks.items()},\n             \'target_disagreements_on_usable_rows\': mismatch,\n             \'inherited_limitation\': \'Source weather/soil interpolation and backfill cannot be undone from processed parquet. Reanalysis retrospective experiment, not operational validation.\'}\n    norm = dict(median=median, mean=mean, std=std, static_mean=sm, static_std=ss,\n                knots=knots.astype(\'float32\'), threshold=threshold, features=np.array(features),\n                node_ids=np.array(ids), static=s)\n    return dict(x=x, s=s, y=np.nan_to_num(y).astype(\'float32\'), state=state,\n                q=q, dates=dates.to_numpy(\'datetime64[D]\'), ids=ids, basins=nd.basin.to_numpy(),\n                masks=masks, knots=knots.astype(\'float32\'), numeric_dim=len(feature_names),\n                audit=audit, norm=norm)\n\n\nclass PiecewiseEmbedding(nn.Module):\n    """Train-quantile piecewise-linear numeric embeddings (Gorishniy et al.)."""\n    def __init__(self, knots, dim):\n        super().__init__()\n        self.register_buffer(\'left\', torch.as_tensor(knots[:, :-1]).float())\n        self.register_buffer(\'width\', torch.as_tensor(np.diff(knots)).float().clamp_min(1e-5))\n        self.weight = nn.Parameter(torch.randn(*self.left.shape, dim)*0.03)\n        self.bias = nn.Parameter(torch.zeros(self.left.shape[0], dim))\n\n    def forward(self, x):\n        basis = ((x.unsqueeze(-1)-self.left)/self.width).clamp(0,1)\n        return F.gelu(torch.einsum(\'bfi,fie->bfe\', basis, self.weight)+self.bias).flatten(1)\n\n\nclass EnsembleLinear(nn.Module):\n    """Shared matrix and member-specific rank-one input/output modulation.\n\n    Inspired by BatchEnsemble/TabM; this custom temporal network is not the\n    authors\' TabM implementation. Each member receives its own BCE loss.\n    """\n    def __init__(self, din, dout, k):\n        super().__init__()\n        self.linear = nn.Linear(din, dout, bias=False)\n        self.r = nn.Parameter(torch.empty(k,din).bernoulli_(.5)*2-1)\n        self.s = nn.Parameter(torch.ones(k,dout))\n        self.bias = nn.Parameter(torch.zeros(k,dout))\n\n    def forward(self, x):\n        return self.linear(x*self.r)*self.s+self.bias\n\n\nclass HydroTEM(nn.Module):\n    def __init__(self, data, cfg, temporal=True):\n        super().__init__()\n        self.k, self.numeric_dim, self.temporal = cfg.members, data[\'numeric_dim\'], temporal\n        n = data[\'x\'].shape[-1]\n        self.embed = PiecewiseEmbedding(data[\'knots\'], cfg.embedding)\n        din = self.numeric_dim*cfg.embedding + n + data[\'s\'].shape[-1]\n        self.current = nn.Sequential(nn.Linear(din,cfg.width), nn.GELU(), nn.LayerNorm(cfg.width))\n        if temporal:\n            self.history = nn.Sequential(nn.Linear(n*4,cfg.width), nn.GELU(),\n                                         nn.Dropout(cfg.dropout), nn.Linear(cfg.width,cfg.width))\n            # Gate starts almost closed: protect the strong current-day branch.\n            self.gate = nn.Parameter(torch.full((cfg.width,), -2.0))\n        self.layers = nn.ModuleList([EnsembleLinear(cfg.width,cfg.width,self.k) for _ in range(3)])\n        self.norms = nn.ModuleList([nn.LayerNorm(cfg.width) for _ in range(3)])\n        self.drop = nn.Dropout(cfg.dropout)\n        self.head = EnsembleLinear(cfg.width,4,self.k)\n        with torch.no_grad():\n            # separate dry/wet first-day hazard, then conditional day-2/day-3 hazards\n            self.head.bias[:] = torch.tensor([-4., 1., -4., -4.])\n\n    def forward(self, x, s, state):\n        cur = x[:,-1]\n        h = self.current(torch.cat([self.embed(cur[:,:self.numeric_dim]),cur,s],-1))\n        if self.temporal:\n            # Observed lags and changes, no target-time weather/discharge.\n            history = torch.cat([x[:,-2], x[:,-4:].mean(1), x.mean(1), cur-x[:,0]],-1)\n            h = h + self.gate.sigmoid()*self.history(history)\n        h = h[:,None].expand(-1,self.k,-1)\n        for layer, norm in zip(self.layers,self.norms):\n            h = norm(h+self.drop(F.gelu(layer(h))))\n        z = self.head(h).float()\n        first = z[...,0]*(1-state[:,None]) + z[...,1]*state[:,None]\n        hazards = torch.stack([first,z[...,2],z[...,3]],-1)\n        # Stable cumulative event probabilities: P(any flood within h).\n        p = -torch.expm1(F.logsigmoid(-hazards).cumsum(-1))\n        return torch.cat([p,p[...,:1]*(1-state[:,None,None])],-1)\n\n\nclass Model2Control(nn.Module):\n    def __init__(self, data, cfg):\n        super().__init__()\n        self.net = MMFNet(MMFConfig(n_dynamic=data[\'x\'].shape[-1], n_static=data[\'s\'].shape[-1],\n                                    lookback=cfg.lookback))\n\n    def forward(self, x, s, state):\n        # Existing network uses [snapshot, time, node, feature]; nodes are\n        # independent with graph disabled, so one snapshot is equivalent.\n        out = self.net(x.permute(1,0,2).unsqueeze(0), s)\n        return out[\'logits\'].squeeze(0).sigmoid().unsqueeze(1)\n\n\ndef build_model(name, data, cfg):\n    return Model2Control(data,cfg) if name == \'model2_control\' else HydroTEM(data,cfg, name != \'model4_current\')\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef arrays_at(data, indices, cfg, device):\n    d,n = indices[:,0],indices[:,1]\n    window = d[:,None]-np.arange(cfg.lookback-1,-1,-1)[None]\n    x = torch.as_tensor(data[\'x\'][window,n[:,None]],device=device)\n    s = torch.as_tensor(data[\'s\'][n],device=device)\n    state = torch.as_tensor(data[\'state\'][d,n],device=device,dtype=torch.float32)\n    y = torch.as_tensor(data[\'y\'][d,n],device=device)\n    return x,s,state,y\n\n\ndef objective(p,y):\n    # No focal weighting or oversampling. Each member is supervised separately.\n    losses = F.binary_cross_entropy(p.float().clamp(1e-6,1-1e-6),\n                                   y[:,None].expand_as(p),reduction=\'none\')\n    return (losses*losses.new_tensor([1.,.3,.3,.5])).sum(-1).mean()\n\n\n@torch.no_grad()\ndef predict(model,data,idx,cfg,device):\n    model.eval()\n    result = []\n    for start in range(0,len(idx),cfg.batch_size):\n        x,s,state,_ = arrays_at(data,idx[start:start+cfg.batch_size],cfg,device)\n        with torch.autocast(device_type=device.type, enabled=device.type==\'cuda\'):\n            p = model(x,s,state)\n        result.append(p.float().mean(1).cpu().numpy())\n    return np.concatenate(result)\n\n\ndef ap(y,p):\n    return float(average_precision_score(y,p)) if np.unique(y).size == 2 else float(\'nan\')\n\n\nclass BudgetExpired(Exception):\n    pass\n\n\ndef fit_seed(name,seed,data,cfg,out,deadline,device):\n    seed_all(seed)\n    checkpoint = out/f\'{name}_seed{seed}.pt\'\n    model = build_model(name,data,cfg).to(device)\n    opt = torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)\n    scaler = torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    start,best,bad,history,best_state = 0,-np.inf,0,[],None\n    if checkpoint.exists():\n        saved = torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n        if saved[\'complete\']:\n            model.load_state_dict(saved[\'best_state\'])\n            return model, saved[\'history\']\n        model.load_state_dict(saved[\'state\']); opt.load_state_dict(saved[\'optimizer\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        start,best,bad,history,best_state = (saved[k] for k in [\'epoch\',\'best\',\'bad\',\'history\',\'best_state\'])\n        torch.set_rng_state(saved[\'rng\'])\n        np.random.set_state(saved[\'numpy_rng\']); random.setstate(saved[\'python_rng\'])\n        if device.type==\'cuda\':\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n    train_idx = np.argwhere(data[\'masks\'][\'train\'])\n    val_idx = np.argwhere(data[\'masks\'][\'select\'])\n    target = data[\'y\'][data[\'masks\'][\'select\']]\n    for epoch in range(start,cfg.epochs):\n        tic = time.monotonic()\n        # Reserve enough time for validation and a checkpoint before deadline.\n        if tic+max(60, history[-1][\'seconds\']*1.3 if history else 60) >= deadline:\n            raise BudgetExpired(name)\n        lr = cfg.lr*min(1.,(epoch+1)/5)*(.1+.9*.5*(1+math.cos(math.pi*epoch/cfg.epochs)))\n        for group in opt.param_groups:\n            group[\'lr\']=lr\n        model.train()\n        shuffled = train_idx[np.random.permutation(len(train_idx))]\n        total,seen = 0.,0\n        for i in range(0,len(shuffled),cfg.batch_size):\n            if time.monotonic() >= deadline-30:\n                raise BudgetExpired(name)\n            idx = shuffled[i:i+cfg.batch_size]\n            x,s,state,y = arrays_at(data,idx,cfg,device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=device.type==\'cuda\'):\n                p = model(x,s,state)\n            loss = objective(p,y)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f\'{name} seed {seed}: nonfinite loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip,error_if_nonfinite=True)\n            scaler.step(opt); scaler.update()\n            total += float(loss.detach())*len(idx); seen += len(idx)\n        v = predict(model,data,val_idx,cfg,device)\n        score = ap(target[:,0],v[:,0])\n        if not np.isfinite(score):\n            raise ValueError(\'Selection period needs both positive and negative flood labels.\')\n        if score > best+1e-5:\n            best,bad = score,0\n            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        else:\n            bad += 1\n        history.append(dict(epoch=epoch+1,loss=total/seen,val_ap=score,\n                            val_onset_ap=ap(target[:,3],v[:,3]),lr=lr,seconds=time.monotonic()-tic))\n        complete = bad>=cfg.patience or epoch+1==cfg.epochs\n        torch_save(checkpoint,dict(state=model.state_dict(),best_state=best_state,optimizer=opt.state_dict(),\n                                  scaler=scaler.state_dict(),epoch=epoch+1,best=best,bad=bad,history=history,\n                                  complete=complete,rng=torch.get_rng_state(),numpy_rng=np.random.get_state(),\n                                  python_rng=random.getstate(),cuda_rng=torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []))\n        print(f\'{name} seed={seed} epoch={epoch+1:02d} loss={total/seen:.5f} val_AP={score:.4f} best={best:.4f} ({time.monotonic()-tic:.0f}s)\',flush=True)\n        if complete:\n            break\n    model.load_state_dict(best_state)\n    return model,history\n\n\ndef fit_calibrator(y,p):\n    """Positive-slope logit scaling fitted exclusively on 2020.\n\n    One common mapping across the flood horizons preserves their ordering.\n    Onset is mapped separately for controls and derived exactly for Model 4.\n    """\n    if np.unique(y).size < 2:\n        return [1.,0.]\n    z = np.log(np.clip(p,1e-6,1-1e-6)/np.clip(1-p,1e-6,1))\n    def loss(ab):\n        zz = ab[0]*z+ab[1]\n        return np.mean(np.logaddexp(0,zz)-y*zz)\n    fit = minimize(loss,[1.,0.],method=\'L-BFGS-B\',bounds=[(.05,10),(-10,10)])\n    return fit.x.tolist() if fit.success else [1.,0.]\n\n\ndef calibrate(p,ab):\n    z = np.log(np.clip(p,1e-6,1-1e-6)/np.clip(1-p,1e-6,1))\n    return 1/(1+np.exp(-np.clip(ab[0]*z+ab[1],-50,50)))\n\n\ndef choose_threshold(y,p,max_far=None):\n    precision,recall,threshold = precision_recall_curve(y,p)\n    score = recall[:-1] if max_far is not None else 2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)\n    ok = np.ones(len(threshold),bool) if max_far is None else (1-precision[:-1]<=max_far)\n    if not ok.any():\n        return 1.000001  # Explicit no-feasible-alarm policy, never silently .5.\n    score = np.where(ok,score,-np.inf)\n    return float(threshold[np.argmax(score)])\n\n\ndef metrics(y,p,threshold):\n    alarms = p>=threshold\n    tp,fp,fn = int(np.sum(alarms & (y==1))),int(np.sum(alarms & (y==0))),int(np.sum(~alarms & (y==1)))\n    ece = 0.\n    for k in range(15):\n        m = np.minimum((p*15).astype(int),14)==k\n        if m.any():\n            ece += m.mean()*abs(y[m].mean()-p[m].mean())\n    return dict(pr_auc=ap(y,p),roc_auc=float(roc_auc_score(y,p)) if np.unique(y).size==2 else None,\n                brier=float(np.mean((p-y)**2)),ece=float(ece),n=len(y),positive_rate=float(y.mean()),\n                threshold=threshold,tp=tp,fp=fp,fn=fn,pod=tp/max(tp+fn,1),far=fp/max(tp+fp,1),\n                csi=tp/max(tp+fp+fn,1),f1=2*tp/max(2*tp+fp+fn,1))\n\n\ndef event_report(data,idx,p,threshold,lead=3):\n    """True full-panel starts, complete forecast windows, currently dry alarms.\n\n    A 3-day merged episode separates events by at least three dry days. This\n    shares the catalogue\'s <=2-day flood-to-flood gap convention.\n    """\n    lookup = {(int(d),int(n)):float(prob) for (d,n),prob in zip(idx,p)}\n    detected,total,leads = 0,0,[]\n    for n in range(len(data[\'ids\'])):\n        wet = np.flatnonzero(data[\'state\'][:,n]==1)\n        starts = wet[np.r_[True,np.diff(wet)>2]] if wet.size else []\n        for d in starts:\n            window = range(d-lead,d)\n            if d<lead or not all((t,n) in lookup for t in window):\n                continue\n            total += 1\n            hit = [d-t for t in window if data[\'state\'][t,n]==0 and lookup[t,n]>=threshold]\n            if hit:\n                detected += 1; leads.append(max(hit))\n    return dict(n_eligible_events=total,n_detected=detected,event_detection_rate=detected/max(total,1),\n                mean_lead_days=float(np.mean(leads)) if leads else None,max_lead_days=lead)\n\n\ndef paired_bootstrap(y,a,b,dates,draws):\n    # Resample contiguous 28-day blocks; keep ALL nodes in each time block.\n    blocks = (dates-dates.min()).astype(\'timedelta64[D]\').astype(int)//28\n    unique = np.unique(blocks)\n    rng = np.random.default_rng(712)\n    diffs = []\n    # Sample weights avoid copying or concatenating hundreds of thousands of rows.\n    _,codes = np.unique(blocks,return_inverse=True)\n    for _ in range(draws):\n        counts = np.bincount(rng.integers(0,len(unique),len(unique)),minlength=len(unique))\n        w = counts[codes]\n        if np.sum(w*y)==0 or np.sum(w*(1-y))==0:\n            continue\n        diffs.append(average_precision_score(y,a,sample_weight=w)-average_precision_score(y,b,sample_weight=w))\n    return dict(delta_ap=ap(y,a)-ap(y,b),ci95=np.quantile(diffs,[.025,.975]).tolist() if diffs else None,\n                draws=len(diffs),block_days=28,spatial_unit=\'all nodes jointly\',\n                caveat=\'Conditional on fitted models; does not include training uncertainty or test-reuse selection bias.\')\n\n\ndef baseline_predictions(data,cfg,out):\n    from lightgbm import LGBMClassifier, early_stopping, log_evaluation\n    import joblib\n    indices = {k:np.argwhere(v) for k,v in data[\'masks\'].items()}\n    def features(idx):\n        d,n = idx.T\n        return np.concatenate([data[\'x\'][d,n],data[\'s\'][n],\n                               data[\'x\'][d-1,n],data[\'x\'][d-3,n],data[\'x\'][d,n]-data[\'x\'][d-7,n]],-1)\n    xx = {k:features(v) for k,v in indices.items()}\n    yy = {k:data[\'y\'][data[\'masks\'][k]] for k in indices}\n    predictions = {name:{k:[] for k in [\'calibrate\',\'test\']} for name in [\'lightgbm\',\'persistence\',\'discharge_pctl\']}\n    for h in range(4):\n        tree = LGBMClassifier(n_estimators=1500,learning_rate=.025,num_leaves=31,min_child_samples=100,\n                              metric=\'average_precision\',\n                              colsample_bytree=.9,reg_lambda=2.,n_jobs=4,random_state=0,verbosity=-1)\n        tree.fit(xx[\'train\'],yy[\'train\'][:,h],eval_set=[(xx[\'select\'],yy[\'select\'][:,h])],\n                 eval_metric=\'average_precision\',callbacks=[early_stopping(75,first_metric_only=True),log_evaluation(0)])\n        joblib.dump(tree,out/f\'lightgbm_head{h}.joblib\')\n        for k in [\'calibrate\',\'test\']:\n            d,n = indices[k].T\n            predictions[\'lightgbm\'][k].append(tree.predict_proba(xx[k])[:,1])\n            # Persistent state predicts no onset. This is the actual baseline.\n            predictions[\'persistence\'][k].append(data[\'state\'][d,n] if h<3 else np.zeros(len(d)))\n            pos = DYNAMIC_FEATURES.index(\'discharge_pctl\')\n            percentile = data[\'x\'][d,n,pos]*data[\'norm\'][\'std\'][pos]+data[\'norm\'][\'mean\'][pos]\n            predictions[\'discharge_pctl\'][k].append(percentile if h<3 else percentile*(1-data[\'state\'][d,n]))\n    return {name:{k:np.stack(v,-1) for k,v in pp.items()} for name,pp in predictions.items()}\n\n\ndef evaluate_all(predictions,data,cfg,out,seed_metrics):\n    ci = np.argwhere(data[\'masks\'][\'calibrate\']); ti = np.argwhere(data[\'masks\'][\'test\'])\n    cy = data[\'y\'][data[\'masks\'][\'calibrate\']]; ty = data[\'y\'][data[\'masks\'][\'test\']]\n    rows,report,calibrated = [],{},{}\n    names = [\'flood_1d\',\'flood_2d\',\'flood_3d\',\'onset_1d\']\n    for name,pp in predictions.items():\n        cp,tp = pp[\'calibrate\'].copy(),pp[\'test\'].copy()\n        # Scaling is fit on calibration only. No test-tuned blend or threshold.\n        ab = fit_calibrator(cy[:,:3].ravel(),cp[:,:3].ravel())\n        cp[:,:3],tp[:,:3] = calibrate(cp[:,:3],ab),calibrate(tp[:,:3],ab)\n        if name.startswith(\'model4\'):\n            cp[:,3]=cp[:,0]*(1-data[\'state\'][tuple(ci.T)])\n            tp[:,3]=tp[:,0]*(1-data[\'state\'][tuple(ti.T)])\n            onset_ab = None\n        elif name==\'persistence\':\n            cp[:,3]=0; tp[:,3]=0; onset_ab=None\n        else:\n            onset_ab = fit_calibrator(cy[:,3],cp[:,3])\n            cp[:,3],tp[:,3]=calibrate(cp[:,3],onset_ab),calibrate(tp[:,3],onset_ab)\n        calibrated[name]=tp\n        seeds = seed_metrics.get(name,[])\n        result = dict(calibration=dict(flood=ab,onset=onset_ab),heads={},seed_metrics=seeds,\n                      n_seeds=len(seeds) if seeds else None,\n                      raw_seed_ap_mean=float(np.mean([s[\'test_ap\'] for s in seeds])) if seeds else None,\n                      raw_seed_ap_std=float(np.std([s[\'test_ap\'] for s in seeds],ddof=1)) if len(seeds)>1 else None)\n        for h,head in enumerate(names):\n            th = choose_threshold(cy[:,h],cp[:,h])\n            far_th = choose_threshold(cy[:,h],cp[:,h],cfg.max_far)\n            result[\'heads\'][head] = dict(at_calibration_f1=metrics(ty[:,h],tp[:,h],th),\n                                        at_calibration_far=metrics(ty[:,h],tp[:,h],far_th),\n                                        raw_ap=ap(ty[:,h],pp[\'test\'][:,h]),\n                                        calibration_far_feasible=far_th<=1)\n            if h in [0,2,3]:\n                result[\'heads\'][head][\'events\'] = event_report(data,ti,tp[:,h],far_th,3 if h==2 else 1)\n            rows.append(dict(model=name,head=head,**result[\'heads\'][head][\'at_calibration_f1\']))\n        # At-risk onset AP removes ongoing floods rather than rewarding easy negatives.\n        dry = data[\'state\'][tuple(ti.T)]==0\n        result[\'onset_at_risk_ap\']=ap(ty[dry,3],tp[dry,3])\n        report[name]=result\n        np.savez_compressed(out/f\'{name}_predictions.npz\',y=ty,p=tp,raw_p=pp[\'test\'],\n                            calibration_y=cy,calibration_p=cp,calibration_indices=ci,\n                            day=ti[:,0],node=ti[:,1],date=data[\'dates\'][ti[:,0]],node_id=np.array(data[\'ids\'])[ti[:,1]])\n    contrasts = {}\n    if \'model4\' in calibrated:\n        for other in [\'model2_control\',\'lightgbm\',\'model4_current\',\'discharge_pctl\']:\n            if other in calibrated:\n                contrasts[other] = paired_bootstrap(ty[:,0],calibrated[\'model4\'][:,0],calibrated[other][:,0],\n                                                    data[\'dates\'][ti[:,0]],cfg.bootstrap_draws)\n    json_save(out/\'metrics.json\',report); json_save(out/\'paired_comparisons.json\',contrasts)\n    pd.DataFrame(rows).to_csv(out/\'summary.csv\',index=False)\n    subgroup = []\n    for name,p in calibrated.items():\n        for group,values in [(\'basin\',data[\'basins\'][ti[:,1]]),(\'year\',data[\'dates\'][ti[:,0]].astype(\'datetime64[Y]\').astype(str))]:\n            for value in np.unique(values):\n                sel = values==value\n                subgroup.append(dict(model=name,group=group,value=value,n=int(sel.sum()),\n                                     flood_ap=ap(ty[sel,0],p[sel,0]),onset_ap=ap(ty[sel,3],p[sel,3])))\n    pd.DataFrame(subgroup).to_csv(out/\'subgroups.csv\',index=False)\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,axes = plt.subplots(1,2,figsize=(12,5))\n    for name,p in calibrated.items():\n        pr,re,_=precision_recall_curve(ty[:,0],p[:,0])\n        axes[0].plot(re,pr,label=f\'{name}: AP {ap(ty[:,0],p[:,0]):.3f}\')\n        order=np.argsort(p[:,0]); bins=np.array_split(order,12)\n        axes[1].plot([p[b,0].mean() for b in bins],[ty[b,0].mean() for b in bins],\'.-\',label=name)\n    axes[0].set(xlabel=\'Recall\',ylabel=\'Precision\',title=\'Held-out 2021–2024: next-day exceedance\')\n    axes[1].plot([0,1],[0,1],\'k--\',alpha=.4)\n    axes[1].set(xlabel=\'Mean predicted probability\',ylabel=\'Observed fraction\',title=\'Reliability (equal-count bins)\')\n    for ax in axes: ax.legend(fontsize=8)\n    fig.tight_layout(); fig.savefig(out/\'evaluation.png\',dpi=170); plt.close(fig)\n    print(pd.DataFrame(rows).query("head == \'flood_1d\'")[[\'model\',\'pr_auc\',\'brier\',\'ece\',\'pod\',\'far\']].to_string(index=False))\n    return report\n\n\ndef run(root,out,cfg):\n    out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    root=Path(root)\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    torch.set_num_threads(min(4,os.cpu_count() or 1))\n    if device.type==\'cuda\':\n        torch.backends.cudnn.benchmark=False\n        torch.backends.cudnn.deterministic=True\n    modules_root=Path(__file__).resolve().parents[1]\n    sources=[\'model4/workflow.py\',\'model2/model.py\',\'model2/modules.py\',\'model2/config.py\',\n             \'floodlib/schema.py\',\'floodlib/blocks.py\',\'floodlib/traincfg.py\']\n    identity=dict(config=asdict(cfg),data_sha256=file_hash(root/\'flood_dataset.parquet\'),\n                  nodes_sha256=file_hash(root/\'nodes.csv\'),\n                  source_sha256={name:file_hash(modules_root/name) for name in sources})\n    signature=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()\n    manifest=out/\'manifest.json\'\n    if manifest.exists() and json.loads(manifest.read_text())[\'signature\']!=signature:\n        raise ValueError(\'Output contains a different dataset/code/config run. Use a fresh output directory.\')\n    json_save(manifest,dict(signature=signature,**identity,device=str(device),torch=torch.__version__,\n                           numpy=np.__version__,python=platform.python_version(),\n                           gpu=torch.cuda.get_device_name(0) if device.type==\'cuda\' else None))\n    data=prepare(pd.read_parquet(root/\'flood_dataset.parquet\'),pd.read_csv(root/\'nodes.csv\'),cfg)\n    json_save(out/\'data_audit.json\',data[\'audit\']); np.savez_compressed(out/\'preprocessing.npz\',**data[\'norm\'])\n    architectures={}\n    for name in [\'model4\',\'model2_control\',\'model4_current\']:\n        model=build_model(name,data,cfg)\n        architectures[name]={\'parameters\':sum(p.numel() for p in model.parameters())}\n        if name==\'model2_control\':\n            architectures[name][\'config\']=asdict(model.net.cfg)\n        del model\n    json_save(out/\'architectures.json\',architectures)\n    print(json.dumps(data[\'audit\'][\'split_counts\']),flush=True)\n    deadline=time.monotonic()+cfg.budget_hours*3600\n    preds={}; seed_metrics={}; pending=[]\n    # Model 4 first, then the paired architecture control, then a cheap ablation.\n    schedule=[(\'model4\',cfg.seeds),(\'model2_control\',cfg.seeds),(\'model4_current\',(cfg.seeds[0],))]\n    for name,seeds in schedule:\n        collected={k:[] for k in [\'calibrate\',\'test\']}; details=[]\n        for seed in seeds:\n            cache=out/f\'{name}_seed{seed}_predictions.npz\'\n            try:\n                if cache.exists():\n                    with np.load(cache) as saved:\n                        pp={k:saved[k] for k in collected}\n                else:\n                    model,history=fit_seed(name,seed,data,cfg,out,deadline,device)\n                    pp={k:predict(model,data,np.argwhere(data[\'masks\'][k]),cfg,device) for k in collected}\n                    np.savez_compressed(cache,**pp)\n                    del model\n                    if device.type==\'cuda\': torch.cuda.empty_cache()\n                for k in collected: collected[k].append(pp[k])\n                yy=data[\'y\'][data[\'masks\'][\'test\']]\n                details.append(dict(seed=seed,test_ap=ap(yy[:,0],pp[\'test\'][:,0]),onset_ap=ap(yy[:,3],pp[\'test\'][:,3])))\n            except BudgetExpired:\n                pending.append(f\'{name}: seed {seed}\')\n                print(f\'Time budget reached; checkpoint retained for {name} seed {seed}.\',flush=True)\n        if len(collected[\'test\'])==len(seeds):\n            preds[name]={k:np.mean(v,axis=0) for k,v in collected.items()}\n            seed_metrics[name]=details\n    # Test metrics are saved only after every planned neural run completes.\n    # This prevents an incomplete seed prefix being presented as the final run.\n    if pending:\n        json_save(out/\'status.json\',dict(complete=False,pending=pending,\n                  resume=\'Rerun with this output directory (or attach extracted outputs as Kaggle input).\'))\n        print(\'PARTIAL: no final test comparison. Resume to complete the predeclared experiment.\',flush=True)\n        return\n    print(\'Neural runs complete. Fitting matched baselines.\',flush=True)\n    base_cache=out/\'baselines.npz\'\n    if base_cache.exists():\n        with np.load(base_cache) as saved:\n            for name in [\'lightgbm\',\'persistence\',\'discharge_pctl\']:\n                preds[name]={k:saved[f\'{name}_{k}\'] for k in [\'calibrate\',\'test\']}\n    else:\n        baselines=baseline_predictions(data,cfg,out); preds.update(baselines)\n        np.savez_compressed(base_cache,**{f\'{name}_{k}\':v for name,pp in baselines.items() for k,v in pp.items()})\n    evaluate_all(preds,data,cfg,out,seed_metrics)\n    json_save(out/\'status.json\',dict(complete=True,pending=[],\n              note=\'Observed test AP is an experiment result, not a guarantee of superiority or operational readiness.\'))\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--root\',required=True)\n    parser.add_argument(\'--out\',default=\'/kaggle/working/model4_runs\')\n    args=parser.parse_args()\n    run(args.root,args.out,Config())\n', 'models/model2/__init__.py': '"""Model 2 — MMF-Net, a graph-free multimodal tokenising transformer.\n\nPeriodic numerical embeddings → temporal transformer over the lookback window\n(+ optional cross-feature attention) → FiLM terrain conditioning → gated fusion\nof a pretrained SAR encoder → multitask heads.\n\nBuilt to answer two things model 1 left open: whether a properly constructed\ntabular deep network can close the gap to gradient-boosted trees, and whether\nthe SAR branch was useless or merely fused badly.\n"""\n', 'models/model2/config.py': '"""Model 2 architecture config and the N0 → N6 preset ladder.\n\nModel 2 is deliberately **graph-free**. Model 1 already answers "does relational\nmessage passing help?", but only against a weak per-node floor (M0, a plain\nGRU). Putting a properly built per-node network on the other side of that\ncomparison is what makes the answer mean something.\n"""\nfrom dataclasses import dataclass, replace\nfrom typing import Dict, Literal, Optional\n\nfrom floodlib.schema import BaseModelConfig, GraphMode\nfrom floodlib.traincfg import Preset, TrainConfig\n\nFAMILY = "MMF-Net"\n\n\n@dataclass\nclass MMFConfig(BaseModelConfig):\n    """Multimodal Flood Net: tokenising temporal transformer + gated SAR fusion."""\n\n    graph_mode: GraphMode = "none"      # no message passing, by design\n\n    # stream 1 — tabular tokenisation\n    num_embed: Literal["linear", "plr"] = "plr"\n    d_emb: int = 8\n    n_freq: int = 8\n    freq_sigma: float = 0.05\n\n    # stream 1 — temporal transformer\n    d_model: int = 128\n    n_layers: int = 3\n    n_heads: int = 4\n    ff_mult: int = 2\n    dropout: float = 0.2\n\n    # stream 2 — cross-feature attention\n    feature_attn: bool = True\n    feature_layers: int = 1\n\n    # stream 4 — SAR\n    sar_pretrained: Optional[str] = None   # path from model2/pretrain_sar.py\n    sar_freeze: bool = True\n\n    # fusion and head — same widths as model 1, so a parameter-count comparison\n    # is about the encoder and not about an arbitrarily wider classifier.\n    fusion_hidden: int = 192\n    fusion_out: int = 128\n    head_hidden: int = 256\n    head_hidden2: int = 128\n\n\ndef _m(**kw) -> MMFConfig:\n    return replace(MMFConfig(), **kw)\n\n\ndef _t(**kw) -> TrainConfig:\n    return replace(TrainConfig(), **kw)\n\n\n#: One change per rung, exactly as the M-ladder. N0 mirrors M0 (no terrain, no\n#: graph, plain BCE) so the two families start from a comparable floor.\nPRESETS: Dict[str, Preset] = {\n    "N0": Preset(\n        "N0", "floor — transformer with plain linear feature embeddings",\n        _m(num_embed="linear", feature_attn=False, static_mode="none",\n           vision_mode="none"),\n        _t(loss="bce"),\n    ),\n    "N1": Preset(\n        "N1", "RQ6 — periodic (PLR) numerical embeddings, the tree-gap fix",\n        _m(num_embed="plr", feature_attn=False, static_mode="none",\n           vision_mode="none"),\n        _t(loss="bce"),\n    ),\n    "N2": Preset(\n        "N2", "RQ7 — cross-feature attention over the 33 channels",\n        _m(num_embed="plr", feature_attn=True, static_mode="none",\n           vision_mode="none"),\n        _t(loss="bce"),\n    ),\n    "N3": Preset(\n        "N3", "RQ4 again — FiLM terrain conditioning, without a graph",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="none"),\n        _t(loss="bce"),\n    ),\n    "N4": Preset(\n        "N4", "class imbalance — focal x label_confidence",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="none"),\n        _t(loss="focal_conf"),\n    ),\n    "N5": Preset(\n        "N5", "5-seed deep ensemble + temperature scaling, on the focal loss",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="none"),\n        _t(loss="focal_conf", n_seeds=5, calibration="temperature"),\n    ),\n    "N5_bce": Preset(\n        # The first ladder2 run showed the focal rung is a regression, not a\n        # gain: N3 (BCE) scored PR-AUC 0.8269 with ECE 0.0032, N4 (focal) 0.7592\n        # with ECE 0.0525, and N5 recovered only to 0.7846. This rung applies the\n        # ensemble and the calibrator to the loss that was actually working, and\n        # is the headline candidate.\n        "N5_bce", "headline candidate — N3 plus 5-seed ensemble + temperature",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="none"),\n        _t(loss="bce", n_seeds=5, calibration="temperature"),\n    ),\n    "N6_scalars": Preset(\n        "N6_scalars", "RQ5 again — SAR scalars appended to the dynamic stream",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="scalars"),\n        _t(loss="focal_conf", n_seeds=5, calibration="temperature"),\n    ),\n    "N6_gated": Preset(\n        "N6_gated", "RQ8 — pretrained SAR encoder fused through a learned gate",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="cnn", image_px=512, sar_freeze=True),\n        _t(loss="focal_conf", n_seeds=5, calibration="temperature", batch_size=8),\n    ),\n    "N6_gated_bce": Preset(\n        # N6_gated (0.8310, 11.97M params) barely cleared N3 (0.8269, 711k\n        # params) -- so the imagery may only be recovering what the focal loss\n        # destroyed. Run against the BCE ladder to find out which it is.\n        "N6_gated_bce", "RQ8 — the gated SAR branch, against the BCE ladder",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="cnn", image_px=512, sar_freeze=True),\n        _t(loss="bce", n_seeds=5, calibration="temperature", batch_size=8),\n    ),\n    "N6_gated_ft": Preset(\n        "N6_gated_ft", "RQ8 — as N6_gated but the encoder is fine-tuned, not frozen",\n        _m(num_embed="plr", feature_attn=True, static_mode="film",\n           vision_mode="cnn", image_px=512, sar_freeze=False),\n        _t(loss="focal_conf", n_seeds=5, calibration="temperature", batch_size=8),\n    ),\n}\n', 'models/model2/modules.py': '"""Model 2\'s layers: numerical embeddings, tokenising transformers, gated fusion.\n\nThe design answers a specific weakness in the model 1 results. On the temporal\nprotocol the gradient-boosted-tree baseline reaches PR-AUC 0.850 while the best\nnetwork reaches 0.742 — the familiar result that trees beat neural networks on\ntabular data. The published fix for that is not a bigger network but a better\ninput layer: give every scalar feature its own learned *embedding* instead of\nfeeding it as one number into a linear layer, and the gap largely closes\n(Gorishniy et al., "On Embeddings for Numerical Features in Tabular Deep\nLearning", NeurIPS 2022). That is what `NumericEmbedding` is.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom typing import Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n# --------------------------------------------------- representation: features\n\nclass NumericEmbedding(nn.Module):\n    """Per-feature embedding of a scalar: [..., F] → [..., F, d_emb].\n\n    Two variants, selected by `kind`:\n\n    ``linear``\n        the plain baseline — feature j is mapped by its own affine map followed\n        by a ReLU. Already stronger than sharing one projection across features,\n        because a millimetre of rain and a soil-wetness fraction get separate\n        parameters.\n    ``plr``\n        periodic-linear-ReLU. Feature j is first expanded into\n        ``[sin(2πc_j x), cos(2πc_j x)]`` over `n_freq` learned frequencies, then\n        projected and rectified. The periodic expansion is what lets a network\n        represent a sharp decision boundary in a scalar — the thing a decision\n        tree gets for free from a split point and a plain MLP struggles with.\n\n    Frequencies are initialised from N(0, σ²) with a small σ: large initial\n    frequencies alias the input and the model never recovers.\n    """\n\n    def __init__(self, n_features: int, d_emb: int = 8, kind: str = "plr",\n                 n_freq: int = 8, sigma: float = 0.05):\n        super().__init__()\n        self.kind = kind\n        d_in = 2 * n_freq if kind == "plr" else 1\n        if kind == "plr":\n            self.coef = nn.Parameter(torch.randn(n_features, n_freq) * sigma)\n        self.weight = nn.Parameter(torch.empty(n_features, d_in, d_emb))\n        self.bias = nn.Parameter(torch.zeros(n_features, d_emb))\n        nn.init.normal_(self.weight, std=d_in ** -0.5)\n        self.d_emb = d_emb\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if self.kind == "plr":\n            v = 2 * math.pi * x.unsqueeze(-1) * self.coef      # [..., F, n_freq]\n            z = torch.cat([torch.sin(v), torch.cos(v)], dim=-1)\n        else:\n            z = x.unsqueeze(-1)                                 # [..., F, 1]\n        return F.relu(torch.einsum("...fi,fio->...fo", z, self.weight) + self.bias)\n\n\n# --------------------------------------------------- architecture: transformers\n\ndef _encoder(d_model: int, n_heads: int, ff_mult: int, dropout: float,\n             layers: int) -> nn.TransformerEncoder:\n    """Pre-LN transformer stack. Pre-LN trains without a warmup-sensitive phase,\n    which matters here because the ladder shares one optimiser schedule."""\n    layer = nn.TransformerEncoderLayer(\n        d_model=d_model, nhead=n_heads, dim_feedforward=ff_mult * d_model,\n        dropout=dropout, activation="gelu", batch_first=True, norm_first=True)\n    return nn.TransformerEncoder(layer, num_layers=layers,\n                                 norm=nn.LayerNorm(d_model))\n\n\nclass TemporalTransformer(nn.Module):\n    """Self-attention over the `lookback` window, one token per day.\n\n    Each day\'s 33 channels are embedded feature-wise and mixed into a single\n    d_model token; a CLS token then pools the window. Compared with the GRU it\n    replaces, attention can look straight back at the day the rain fell rather\n    than carrying it forward through 14 recurrent steps — and the attention row\n    of the CLS token is directly readable as the rainfall→discharge lag.\n    """\n\n    def __init__(self, n_features: int, lookback: int, d_model: int = 128,\n                 d_emb: int = 8, kind: str = "plr", n_freq: int = 8,\n                 sigma: float = 0.05, n_heads: int = 4, ff_mult: int = 2,\n                 layers: int = 3, dropout: float = 0.2):\n        super().__init__()\n        self.embed = NumericEmbedding(n_features, d_emb, kind, n_freq, sigma)\n        self.to_token = nn.Linear(n_features * d_emb, d_model)\n        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))\n        self.pos = nn.Parameter(torch.zeros(1, lookback + 1, d_model))\n        nn.init.trunc_normal_(self.pos, std=0.02)\n        nn.init.trunc_normal_(self.cls, std=0.02)\n        self.enc = _encoder(d_model, n_heads, ff_mult, dropout, layers)\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        """x: [M, L, F] → (pooled [M, d_model], per-feature embeddings [M, L, F, d])."""\n        z = self.embed(x)                                       # [M, L, F, d_emb]\n        tok = self.to_token(z.flatten(-2))                      # [M, L, d_model]\n        tok = torch.cat([self.cls.expand(tok.size(0), -1, -1), tok], dim=1)\n        h = self.enc(self.drop(tok + self.pos[:, : tok.size(1)]))\n        return h[:, 0], z\n\n\nclass FeatureAttention(nn.Module):\n    """Self-attention *across the 33 channels* of a time-averaged window.\n\n    The temporal transformer mixes features through a single linear layer, which\n    cannot express "this much rain matters only when the soil is already wet".\n    Attending over feature tokens can. It is affordable here only because the\n    time axis is averaged out first: 33 tokens per node-day rather than 33 x 14.\n    """\n\n    def __init__(self, n_features: int, d_emb: int, d_model: int,\n                 n_heads: int = 4, ff_mult: int = 2, layers: int = 1,\n                 dropout: float = 0.2):\n        super().__init__()\n        self.proj = nn.Linear(d_emb, d_model)\n        self.feat_tok = nn.Parameter(torch.zeros(1, n_features, d_model))\n        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))\n        nn.init.trunc_normal_(self.feat_tok, std=0.02)\n        nn.init.trunc_normal_(self.cls, std=0.02)\n        self.enc = _encoder(d_model, n_heads, ff_mult, dropout, layers)\n\n    def forward(self, z: torch.Tensor) -> torch.Tensor:\n        """z: [M, L, F, d_emb] → [M, d_model]."""\n        t = self.proj(z.mean(dim=1)) + self.feat_tok            # [M, F, d_model]\n        t = torch.cat([self.cls.expand(t.size(0), -1, -1), t], dim=1)\n        return self.enc(t)[:, 0]\n\n\n# ------------------------------------------------------------- gated fusion\n\nclass GatedSarFusion(nn.Module):\n    """Fuse a SAR embedding into the tabular state through a learned gate.\n\n    Model 1 concatenated the SAR embedding onto the fusion input and the branch\n    returned nothing (RQ5). Concatenation is the wrong operator for this signal:\n    Sentinel-1 revisits every ~12 days, so the overwhelming majority of node-days\n    carry no observation at all, and a fixed slot in a concatenation forces the\n    network to spend capacity distinguishing "no water" from "no picture" on\n    every single sample.\n\n    A gate makes that conditional instead. `g` is computed from the tabular\n    state, the image embedding, the presence flag and the frame\'s age, and the\n    image contributes `g ⊙ W v` — nothing at all when no frame exists. The gate\'s\n    output layer is zero-initialised with a negative bias, so the module starts\n    shut and the SAR rung of the ladder is a strict superset of the rung below it\n    at initialisation.\n    """\n\n    def __init__(self, d_model: int, image_dim: int, init_bias: float = -3.0):\n        super().__init__()\n        self.proj = nn.Linear(image_dim, d_model)\n        self.gate = nn.Sequential(\n            nn.Linear(d_model + image_dim + 2, d_model), nn.GELU(),\n            nn.Linear(d_model, d_model))\n        nn.init.zeros_(self.gate[-1].weight)\n        nn.init.constant_(self.gate[-1].bias, init_bias)\n        self.norm = nn.LayerNorm(d_model)\n        #: Mean gate opening on node-days that actually have a frame, from the\n        #: last forward pass. Read by the diagnostics: a branch whose gate never\n        #: opens contributes nothing, and without this the only evidence would\n        #: be a metric difference too small to attribute to anything.\n        self.last_gate_mean: float = float("nan")\n\n    def forward(self, h: torch.Tensor, v: torch.Tensor, pres: torch.Tensor,\n                age: torch.Tensor) -> torch.Tensor:\n        """h: [B, N, D], v: [B, N, image_dim], pres/age: [B, N]."""\n        pres = pres.unsqueeze(-1)\n        ctx = torch.cat([h, v, pres, (age / 30.0).unsqueeze(-1)], dim=-1)\n        g = torch.sigmoid(self.gate(ctx))\n        with torch.no_grad():\n            denom = pres.sum().clamp_min(1.0) * g.size(-1)\n            self.last_gate_mean = float((g * pres).sum() / denom)\n        return self.norm(h + g * self.proj(v) * pres)\n', 'models/model2/model.py': '"""MMF-Net — model 2, the assembled network.\n\nSame inputs, same targets and same evaluation as model 1; a different answer to\nthe question of how a node\'s own history should be encoded.\n\n    x     [B, L, N, F]   dynamic window (F includes SAR scalars in `scalars` mode)\n    s     [N, S]         static terrain\n    img   [K, C, P, P]   packed SAR frames, only the node-days that have one\n    img_m [B, N]         1 where a frame is present\n\nOutput: logits [B, N, n_cls_heads] and regressions [B, N, n_reg_heads].\n"""\nfrom __future__ import annotations\n\nfrom typing import Dict, Optional\n\nimport torch\nimport torch.nn as nn\n\nfrom floodlib.blocks import FiLM, SarCNN, load_pretrained_encoder\n\nfrom .config import MMFConfig\nfrom .modules import FeatureAttention, GatedSarFusion, TemporalTransformer\n\n\nclass MMFNet(nn.Module):\n    def __init__(self, cfg: MMFConfig, graph=None):\n        super().__init__()\n        self.cfg = cfg\n        if graph is not None and cfg.graph_mode != "none":\n            raise ValueError("model2 is graph-free; set graph_mode=\'none\'")\n\n        self.temporal = TemporalTransformer(\n            n_features=cfg.n_dynamic, lookback=cfg.lookback, d_model=cfg.d_model,\n            d_emb=cfg.d_emb, kind=cfg.num_embed, n_freq=cfg.n_freq,\n            sigma=cfg.freq_sigma, n_heads=cfg.n_heads, ff_mult=cfg.ff_mult,\n            layers=cfg.n_layers, dropout=cfg.dropout)\n\n        self.feat_attn = None\n        if cfg.feature_attn:\n            self.feat_attn = FeatureAttention(\n                cfg.n_dynamic, cfg.d_emb, cfg.d_model, cfg.n_heads,\n                cfg.ff_mult, cfg.feature_layers, cfg.dropout)\n            self.merge = nn.LayerNorm(cfg.d_model)\n\n        fusion_in = cfg.d_model\n        if cfg.static_mode == "film":\n            self.film = FiLM(cfg.n_static, cfg.static_hidden, cfg.d_model)\n        elif cfg.static_mode == "concat":\n            fusion_in += cfg.n_static\n\n        # Unlike model 1, the SAR embedding is gated into the state rather than\n        # concatenated onto it, so it adds no width to the fusion input.\n        self.sar = None\n        if cfg.vision_mode == "cnn":\n            self.cnn = SarCNN(cfg.image_channels, cfg.image_dim)\n            if cfg.sar_pretrained:\n                load_pretrained_encoder(self.cnn, cfg.sar_pretrained,\n                                        freeze=cfg.sar_freeze)\n            self.sar = GatedSarFusion(cfg.d_model, cfg.image_dim)\n\n        self.fuse = nn.Sequential(\n            nn.Linear(fusion_in, cfg.fusion_hidden), nn.GELU(),\n            nn.Linear(cfg.fusion_hidden, cfg.fusion_out), nn.GELU(),\n            nn.LayerNorm(cfg.fusion_out),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(cfg.fusion_out, cfg.head_hidden), nn.GELU(),\n            nn.Dropout(cfg.dropout),\n            nn.Linear(cfg.head_hidden, cfg.head_hidden2), nn.GELU())\n        self.cls = nn.Linear(cfg.head_hidden2, cfg.n_cls_heads)\n        self.reg = nn.Linear(cfg.head_hidden2, cfg.n_reg_heads)\n\n        self.register_buffer("temperature", torch.ones(1), persistent=True)\n\n    # ------------------------------------------------------------------ forward\n    def forward(self, x: torch.Tensor, s: torch.Tensor,\n                img: Optional[torch.Tensor] = None,\n                img_pos: Optional[torch.Tensor] = None,\n                img_mask: Optional[torch.Tensor] = None,\n                img_age: Optional[torch.Tensor] = None,\n                return_attention: bool = False) -> Dict[str, torch.Tensor]:\n        B, L, N, _ = x.shape\n\n        seq = x.permute(0, 2, 1, 3).reshape(B * N, L, -1)\n        h, z = self.temporal(seq)\n        if self.feat_attn is not None:\n            h = self.merge(h + self.feat_attn(z))\n        h = h.view(B, N, -1)\n\n        if self.cfg.static_mode == "film":\n            h = self.film(h, s)\n        elif self.cfg.static_mode == "concat":\n            h = torch.cat([h, s.unsqueeze(0).expand(B, -1, -1)], dim=-1)\n\n        if self.sar is not None:\n            flat = h.new_zeros(B * N, self.cfg.image_dim)\n            if img is not None and img.numel() and img_pos is not None:\n                flat.index_copy_(0, img_pos, self.cnn(img))\n            pres = img_mask if img_mask is not None else h.new_zeros(B, N)\n            age = img_age if img_age is not None else h.new_zeros(B, N)\n            h = self.sar(h, flat.view(B, N, -1), pres, age)\n\n        feat = self.head(self.fuse(h))\n        out = {"logits": self.cls(feat), "reg": self.reg(feat)}\n        if return_attention:\n            out["temporal_attention"] = None\n        return out\n\n    # -------------------------------------------------------------- inference\n    @torch.no_grad()\n    def predict_proba(self, *args, **kw) -> torch.Tensor:\n        """Calibrated probabilities for the primary head (temperature applied)."""\n        logits = self.forward(*args, **kw)["logits"]\n        return torch.sigmoid(logits / self.temperature.clamp_min(1e-3))\n\n    def n_params(self) -> int:\n        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n', 'models/floodlib/__init__.py': '"""Shared library behind every model family in this repo.\n\n`floodlib` owns the data and the evaluation; a model package owns only its\narchitecture. The split is what keeps `model1` (TF-STGNN, a relational graph\nnetwork) and `model2` (MMF-Net, a tokenising transformer) comparable: they read\nthe same 33 channels, use the same splits, and are scored by the same metrics\ncode, so a difference in their result rows is a difference in architecture.\n\n    floodlib.schema     column contract, modes, BaseModelConfig\n    floodlib.traincfg   TrainConfig, Preset\n    floodlib.data       parquet → dense [T, N, F] panel + SnapshotBatcher\n    floodlib.graph      river graph (used by model1 only)\n    floodlib.sar        Sentinel-1 join, frame store, scalar channels\n    floodlib.blocks     layers shared by both families (GRU, FiLM, SarCNN)\n    floodlib.losses     multi-head focal / BCE loss\n    floodlib.metrics    PR-AUC, Brier decomposition, ECE, event detection\n    floodlib.calibrate  temperature and isotonic calibration\n    floodlib.engine     the training loop both families run through\n    floodlib.baselines  the four non-neural reference models\n"""\n\n__all__ = [\n    "schema", "traincfg", "data", "graph", "sar", "blocks",\n    "losses", "metrics", "calibrate", "engine", "baselines",\n]\n', 'models/floodlib/schema.py': '"""Dataset schema — the contract every model family shares.\n\nThis module holds the things that describe *the data*, not *a model*: which\ncolumns are read, which are log-scaled, which targets are predicted, and the\ncore configuration fields the training engine needs regardless of architecture.\n\nIt lives in `floodlib` rather than in a model package on purpose. If `model1`\nand `model2` disagreed about which 33 channels are the input or which column is\nthe primary target, their result tables would not be comparable, and comparing\nthem is the whole point of having two.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Literal\n\n# ---------------------------------------------------------------- feature sets\n\nDYNAMIC_FEATURES = [\n    "precipitation_sum", "precip_sum_2d", "precip_sum_3d",\n    "precip_sum_5d", "precip_sum_7d", "precip_sum_15d", "precip_sum_30d",\n    "precip_max_3d", "precip_max_7d", "api_k090", "wetdays_7d",\n    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",\n    "relative_humidity_2m", "windspeed_10m_mean", "windspeed_10m_max",\n    "shortwave_radiation",\n    "soil_wet_top", "soil_wet_root", "soil_wet_profile",\n    "soil_wet_top_anom", "soil_wet_root_anom", "soil_wet_profile_anom",\n    "discharge", "log_discharge", "discharge_rise_1d", "discharge_rise_3d",\n    "discharge_mean_3d", "discharge_mean_7d", "discharge_anom",\n    "discharge_zscore", "discharge_pctl",\n]  # 33 dynamic channels\n\n# Heavy-tailed columns get log1p before z-scoring (§7.7 "Normalisation").\nLOG1P_FEATURES = {\n    "precipitation_sum", "precip_sum_2d", "precip_sum_3d", "precip_sum_5d",\n    "precip_sum_7d", "precip_sum_15d", "precip_sum_30d",\n    "precip_max_3d", "precip_max_7d", "api_k090",\n    "discharge", "discharge_mean_3d", "discharge_mean_7d",\n}\n\n# Static terrain: 2 continuous + zone one-hot (3) + position one-hot (4) = 9.\nSTATIC_CONTINUOUS = ["elevation_m", "log_drainage_proxy"]\nZONES = ["wet", "intermediate", "dry"]\nPOSITIONS = ["upstream", "mid", "downstream", "outlet"]\nSTATIC_DIM = len(STATIC_CONTINUOUS) + len(ZONES) + len(POSITIONS)\n\n# Classification heads, in fixed order. Index 0 is the primary target.\nCLS_HEADS = ["target_flood_1d", "target_flood_2d", "target_flood_3d", "target_onset_1d"]\n# Regression heads: next-day discharge percentile, forward 3-day max discharge z-score.\nREG_HEADS = ["reg_discharge_pctl_1d", "reg_discharge_z_max_3d"]\n\n# Per-frame SAR scalars used by the `scalars` vision mode (§7.7 stream 3).\nSAR_SCALARS = ["water_fraction", "vv_mean", "vh_mean", "valid_fraction"]\n\n\n# ------------------------------------------------------------------ modes\n\nGraphMode = Literal["none", "spatial", "flow", "both"]\nStaticMode = Literal["none", "concat", "film"]\nVisionMode = Literal["none", "scalars", "cnn"]\nLossName = Literal["bce", "wbce", "focal", "focal_conf"]\nProtocol = Literal["temporal", "basin", "event", "random"]\n\n\n# ------------------------------------------------------------ base model config\n\n@dataclass\nclass BaseModelConfig:\n    """Fields the training engine reads out of *any* architecture\'s config.\n\n    A model family subclasses this and adds its own hyperparameters. The engine\n    never touches those — it only needs to know how long a window is, which\n    modalities to prepare, and how many heads to expect.\n    """\n    lookback: int = 14\n    n_dynamic: int = len(DYNAMIC_FEATURES)\n    n_static: int = STATIC_DIM\n\n    static_mode: StaticMode = "film"\n    static_hidden: int = 64\n\n    vision_mode: VisionMode = "none"\n    image_dim: int = 64\n    image_px: int = 256          # 512 for the full-res Kaggle frames\n    image_channels: int = 2      # VV, VH\n\n    graph_mode: GraphMode = "none"\n\n    n_cls_heads: int = len(CLS_HEADS)\n    n_reg_heads: int = len(REG_HEADS)\n', 'models/floodlib/traincfg.py': '"""Training configuration and the preset container.\n\nShared by every model family: the optimiser, the loss, the split protocol and\nthe calibration policy are properties of *the experiment*, not of the network.\nHolding them here is what makes an M-row and an N-row in the results table an\napples-to-apples comparison.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Dict, Literal, Optional\n\nfrom .schema import BaseModelConfig, LossName, Protocol\n\n\n@dataclass\nclass TrainConfig:\n    protocol: Protocol = "temporal"\n    loss: LossName = "focal_conf"\n    focal_alpha: float = 0.75\n    focal_gamma: float = 2.0\n    pos_weight: Optional[float] = None       # used by `wbce` only\n\n    head_weights: Dict[str, float] = field(default_factory=lambda: {\n        "target_flood_1d": 1.0,\n        "target_flood_2d": 0.3,\n        "target_flood_3d": 0.3,\n        "target_onset_1d": 0.5,\n    })\n    reg_weight: float = 0.2\n\n    #: Which classification head the run is *scored* on — an index into\n    #: `schema.CLS_HEADS`, where 0 is `target_flood_1d` and 3 is\n    #: `target_onset_1d`. Training still optimises all four heads under\n    #: `head_weights`; this selects only what the metrics, the early-stopping\n    #: signal and the saved `_preds.npz` refer to.\n    #:\n    #: It exists because PR-AUC on `target_flood_1d` is near-saturated by\n    #: discharge autocorrelation — the `discharge_pctl` baseline alone reaches\n    #: 0.816 — so that head measures persistence more than skill. Onset is the\n    #: head on which every baseline actually fails. Defaults to 0, so every\n    #: model 1 and model 2 number already published is reproduced unchanged.\n    eval_head: int = 0\n\n    epochs: int = 60\n    warmup_epochs: int = 5\n    lr: float = 1e-3\n    weight_decay: float = 1e-4\n    batch_size: int = 32          # full-graph day snapshots\n    grad_clip: float = 1.0\n    patience: int = 10            # early stopping on val event-level PR-AUC\n    select_metric: str = "pr_auc"\n\n    seed: int = 0\n    n_seeds: int = 1              # >1 → deep ensemble\n    calibration: Literal["none", "temperature", "isotonic"] = "none"\n\n    device: str = "auto"\n    num_workers: int = 0\n    truncate_after: Optional[str] = "2024-12-31"   # §11.5 ragged-2025 policy\n    log_every: int = 1\n\n\n@dataclass\nclass Preset:\n    """One rung of a build ladder: a name, what it answers, and two configs."""\n    name: str\n    answers: str\n    model: BaseModelConfig\n    train: TrainConfig\n', 'models/floodlib/blocks.py': '"""Neural building blocks used by more than one model family.\n\nAnything architecture-defining lives in the model package that defines it — the\nrelational GATv2 in `model1`, the tokenising transformer in `model2`. What is\nhere is shared machinery, so that a difference between the two families is a\ndifference of architecture and never an accident of two slightly different\nimplementations of the same layer.\n"""\nfrom __future__ import annotations\n\nfrom typing import Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n# ------------------------------------------------------------ temporal (GRU)\n\nclass TemporalEncoder(nn.Module):\n    """2-layer GRU over the lookback window + learned attention pooling.\n\n    The attention weights are returned as well: averaged per basin they show the\n    rainfall→discharge lag, which is the interpretability figure promised in the\n    proposal.\n    """\n\n    def __init__(self, n_in: int, hidden: int = 128, layers: int = 2,\n                 dropout: float = 0.2, pool: str = "attention"):\n        super().__init__()\n        self.gru = nn.GRU(n_in, hidden, num_layers=layers, batch_first=True,\n                          dropout=dropout if layers > 1 else 0.0)\n        self.pool = pool\n        if pool == "attention":\n            self.att = nn.Sequential(nn.Linear(hidden, hidden), nn.Tanh(),\n                                     nn.Linear(hidden, 1))\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:\n        """x: [M, L, F] → ([M, H], attention [M, L] or None)."""\n        h, _ = self.gru(x)\n        if self.pool == "last":\n            return self.drop(h[:, -1]), None\n        if self.pool == "mean":\n            return self.drop(h.mean(1)), None\n        a = torch.softmax(self.att(h).squeeze(-1), dim=1)      # [M, L]\n        return self.drop((h * a.unsqueeze(-1)).sum(1)), a\n\n\n# ------------------------------------------------------------------- terrain\n\nclass FiLM(nn.Module):\n    """Static terrain modulates the temporal state: h ← γ ⊙ h + β, then LayerNorm.\n\n    γ is parameterised as 1 + Δγ so the module starts at identity, which keeps\n    the conditioned model a strict superset of the unconditioned one at\n    initialisation.\n    """\n\n    def __init__(self, n_static: int, hidden: int, feat: int):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(n_static, hidden), nn.ReLU(),\n                                 nn.Linear(hidden, 2 * feat))\n        nn.init.zeros_(self.net[-1].weight)\n        nn.init.zeros_(self.net[-1].bias)\n        self.norm = nn.LayerNorm(feat)\n\n    def forward(self, h: torch.Tensor, s: torch.Tensor) -> torch.Tensor:\n        """h: [B, N, D], s: [N, n_static] → [B, N, D]."""\n        gamma, beta = self.net(s).chunk(2, dim=-1)             # [N, D] each\n        return self.norm(h * (1.0 + gamma.unsqueeze(0)) + beta.unsqueeze(0))\n\n\n# -------------------------------------------------------------------- vision\n\nclass BasicBlock(nn.Module):\n    def __init__(self, cin: int, cout: int, stride: int = 1):\n        super().__init__()\n        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)\n        self.b1 = nn.BatchNorm2d(cout)\n        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)\n        self.b2 = nn.BatchNorm2d(cout)\n        self.short = (nn.Sequential() if stride == 1 and cin == cout else\n                      nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),\n                                    nn.BatchNorm2d(cout)))\n\n    def forward(self, x):\n        y = F.relu(self.b1(self.c1(x)), inplace=True)\n        y = self.b2(self.c2(y))\n        return F.relu(y + self.short(x), inplace=True)\n\n\nclass SarCNN(nn.Module):\n    """ResNet-18 with a 2-channel stem (VV, VH), trained from scratch.\n\n    Pretrained ImageNet weights are deliberately not used: SAR backscatter in dB\n    has nothing in common with RGB natural-image statistics, and the 2-channel\n    stem would have to be re-initialised anyway. `model2` instead pretrains this\n    same encoder on the labelled flood/dry chips — see `model2/pretrain_sar.py`.\n    """\n\n    def __init__(self, in_ch: int = 2, out_dim: int = 64, width: int = 64):\n        super().__init__()\n        w = width\n        self.stem = nn.Sequential(\n            nn.Conv2d(in_ch, w, 7, 2, 3, bias=False), nn.BatchNorm2d(w),\n            nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))\n        cfg = [(w, w, 1), (w, w, 1), (w, 2 * w, 2), (2 * w, 2 * w, 1),\n               (2 * w, 4 * w, 2), (4 * w, 4 * w, 1), (4 * w, 8 * w, 2), (8 * w, 8 * w, 1)]\n        self.blocks = nn.Sequential(*[BasicBlock(a, b, s) for a, b, s in cfg])\n        self.head = nn.Linear(8 * w, out_dim)\n        self.frozen_bn = False\n\n    def train(self, mode: bool = True):\n        """Keep BatchNorm in eval mode when the encoder is frozen.\n\n        Freezing the weights is not enough on its own: BatchNorm would still\n        update its running statistics on every forward pass, so a "frozen"\n        feature extractor would quietly drift away from the representation it\n        was pretrained to produce.\n        """\n        super().train(mode)\n        if self.frozen_bn:\n            for m in self.modules():\n                if isinstance(m, nn.BatchNorm2d):\n                    m.eval()\n        return self\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        """x: [M, 2, H, W] → [M, out_dim]."""\n        z = self.blocks(self.stem(x))\n        return self.head(F.adaptive_avg_pool2d(z, 1).flatten(1))\n\n\ndef load_pretrained_encoder(cnn: SarCNN, path: str, freeze: bool = True,\n                            verbose: bool = True) -> SarCNN:\n    """Load `pretrain_sar.py` weights into a `SarCNN`, tolerating a head mismatch.\n\n    The pretraining task has a 2-way classifier on top; the projection head that\n    feeds the fusion layer is a different shape, so only stem+blocks transfer.\n    """\n    state = torch.load(path, map_location="cpu")\n    state = state.get("encoder", state)\n    keep = {k: v for k, v in state.items()\n            if k in cnn.state_dict() and cnn.state_dict()[k].shape == v.shape}\n    missing = [k for k in cnn.state_dict() if k not in keep]\n    cnn.load_state_dict(keep, strict=False)\n    if freeze:\n        for name, p in cnn.named_parameters():\n            p.requires_grad = name.startswith("head")\n        cnn.frozen_bn = True\n        cnn.eval()\n    if verbose:\n        print(f"[sar-pretrain] loaded {len(keep)}/{len(cnn.state_dict())} tensors "\n              f"from {path} (not loaded: {len(missing)}) | "\n              f"{\'frozen\' if freeze else \'fine-tuning\'}")\n    return cnn\n'}
for relative, source in SOURCES.items():
    path = CODE_ROOT/relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE_ROOT/'models'))
print('Embedded modules:', len(SOURCES))


In [ ]:
import shutil
import traceback
from model4.workflow import Config, run

OUT = Path('/kaggle/working/model4_runs')
OUT.mkdir(exist_ok=True)
# To resume a previous session, attach its extracted model4_runs output folder.
# Only checkpoint files generated by your own previous run should be attached.
if not (OUT/'manifest.json').exists():
    candidates = list(Path('/kaggle/input').rglob('model4_runs/manifest.json'))
    if len(candidates)==1:
        shutil.copytree(candidates[0].parent, OUT, dirs_exist_ok=True)
        print('Restored previous run:', candidates[0].parent)
    elif len(candidates)>1:
        raise RuntimeError('Multiple resumable outputs found; attach one previous model4_runs folder.')

CFG = Config()  # Complete default experiment; no user edits needed.
try:
    run(DATA_ROOT, OUT, CFG)
except Exception:
    (OUT/'failure.txt').write_text(traceback.format_exc())
    raise
finally:
    # Always preserve finished seeds and epoch checkpoints, including on error.
    shutil.copytree(CODE_ROOT, OUT/'source', dirs_exist_ok=True)
    with (OUT/'environment.txt').open('w') as environment:
        subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=environment, check=False)
    archive = shutil.make_archive('/kaggle/working/model4_results', 'zip', OUT.parent, OUT.name)
    print('Download:', archive)


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image, FileLink

status = json.loads((OUT/'status.json').read_text())
display(status)
if status['complete']:
    results = pd.read_csv(OUT/'summary.csv')
    display(results[['model','head','pr_auc','brier','ece','pod','far','csi']])
    display(Image(filename=str(OUT/'evaluation.png')))
    display(json.loads((OUT/'paired_comparisons.json').read_text()))
else:
    print('Partial run. Checkpoints are saved; attach extracted output to a new session and Run All to resume.')
display(FileLink('/kaggle/working/model4_results.zip'))


## Outputs and next decisions

Save the notebook version to retain `/kaggle/working/model4_results.zip`.
It includes model/optimizer/RNG checkpoints, exact source and preprocessing,
dataset hashes, training histories, all four heads' raw/calibrated predictions,
per-seed results, validation-fitted thresholds, basin/year diagnostics, PR and
reliability plots, and paired confidence intervals against the new controls.

The 7.5-hour training budget leaves time for final evaluation and packaging.
If training is incomplete, `status.json` names pending work and **no final test
comparison is issued**. Attach the extracted `model4_runs` directory from the
previous output to resume. Code/config/data hashes prevent mixing experiments.
Budget is cooperative, not a guarantee against platform termination.

Read the fresh Model 4 vs Model 2 and LightGBM intervals before claiming a gain.
Intervals condition on trained models and use 28-day time blocks with all nodes
together; they do not establish independence of long events. Thresholds target
FAR ≤ 0.231 on calibration data; achieved test FAR may differ. Onset AP is
reported both over all samples and only currently dry samples. Event lead
windows are one day for 24h/onset heads and three days for the 72h head.
